In [1]:
# 1-dataset model (HTCas9)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = -0.21557972678904422
../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.2323603852247398
../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.322735007922333
../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.21682380262034406
../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.015284877720501189
../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = -0.13464495926874362
../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = -0.23332567770664284
../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.011363910750615623
../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.09509716277025712
../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = -0.2893858465386101


In [3]:
# 2-dataset model (HTCas9+HT11)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.012914507179344179
../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.31879673165652805
../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = -0.00760147993121871
../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.20402142803013645
../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.16270649949904628
../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = -0.18542486741940664
../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.07116469625594027
../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.13009729681560275
../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.3639655783106346
../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0.2517133318439759


In [5]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.2995362857367792
../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.2208414716153232
../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.13963373127372744
../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.1451926894614966
../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.2153585664872049
../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.16841375560789063
../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.19389660454154348
../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.1965874907914423
../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.3950679625689614
../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.2904808508132064


In [7]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.606680540040649
../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.5861393349834655
../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.5634996855758548
../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.4594943473540356
../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.5537743937107993
../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.5541006959736967
../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.41955906744598837
../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.4490743851135124
../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.33899651096122974
../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.41745740481718574


In [9]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]

    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.4516823782715604
../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.3847759489601379
../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.3910970894099362
../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.6143817262390995
../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.5918799585402172
../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.5304579424299455
../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.5633131831166834
../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.6205049773576203
../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.5656594313304423
../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.5817095543497173


In [11]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_change_seq():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_change_seq():
    with open('normalized_change_seq_log_transformed_20nt_sampled_2.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_change_seq = load_branch1_data_change_seq()
    X1_change_seq   = np.asarray(X1_change_seq)
    X1 = np.concatenate([X1_change_seq], axis=0) 

    rates_change_seq = load_reaction_rates_change_seq()
    rates_change_seq   = np.asarray(rates_change_seq)
    rates = np.concatenate([rates_change_seq], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen change_seq dataset
    np.random.seed(42)
    full_indices_change_seq = np.arange(len(rates_change_seq))
    selected_indices_change_seq = np.random.choice(len(full_indices_change_seq), size=len(full_indices_change_seq), replace=False)
    unseen_indices_change_seq = np.setdiff1d(full_indices_change_seq, selected_indices_change_seq)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_change_seq = Subset(hybrid_dataset, unseen_indices_change_seq)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_change_seq = Subset(hybrid_dataset, selected_indices_change_seq)
    trial_loader = DataLoader(selected_set_change_seq, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.3611829969664115
../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.48609484757290866
../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.595263429153072
../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.5061652304171886
../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.5957586200647831
../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.5930460699760327
../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.5933138476207629
../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.5954042563718313
../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.5915090694471843
../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.5578290899889663
